# GraphPack experiments — Gemma 4 Developer Agent Paper Track

Reproduces every number in the paper: localization on the competition's public dev tasks
with Gemma 4 E4B / E2B under fixed token budgets.

**Before running:** Settings → Accelerator **GPU L4x4**, Internet **off** (L4 sessions must be offline;
graphpack itself needs only Python + NumPy). Add Input → the competition dataset, the Gemma 4
model(s) (`gemma-4-e4b-it-qat-w4a16-ct`, `gemma-4-e2b-it-qat-w4a16-ct`), the `graphpack-code`
dataset, and whatever the competition's *Getting Started* notebook attaches to provide vLLM offline.
Results land in `/kaggle/working/results/` and `/kaggle/working/report/`.

In [ ]:
import os, sys, glob, subprocess, time, json
# where the graphpack code is (Kaggle Dataset upload or a git clone)
def find(tail, roots=('/kaggle/input', '/kaggle/working'), depth=6):
    # shallow glob, level by level: a recursive ** over the 20+ GB competition data is slow
    out = []
    for root in roots:
        for d in range(depth):
            out += glob.glob('/'.join([root] + ['*'] * d + [tail]))
    return sorted(set(out))
# finds graphpack/ wherever Kaggle mounted the uploaded code dataset
found = find('graphpack/__init__.py')
assert found, 'add the graphpack code as an input dataset (folder containing graphpack/)'
CODE_DIR = os.path.dirname(os.path.dirname(found[0]))
sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR   # for the `!python -m graphpack...` subprocesses
os.chdir('/kaggle/working')
os.makedirs('results', exist_ok=True)
print(CODE_DIR)
!python {CODE_DIR}/tests/test_smoke.py

## 1. Look at the competition data, then build `tasks.jsonl`
Each task needs: an id, the issue text, the gold patch, and a directory with the **pre-patch** repository.
Run the next cell and send its whole output; then set `SRC` / `REPO_TEMPLATE` / field names below.

In [ ]:
# Shows what the competition data looks like. Copy ALL of this cell's output and send it.
COMP = [p for p in find('*gemma-4-developer-agent*', roots=('/kaggle/input',), depth=3) if os.path.isdir(p)]
print('COMP =', COMP)
ROOT = COMP[0] if COMP else '/kaggle/input'
def shallow(root, depth=3, per_dir=12):
    base = root.rstrip('/').count('/')
    for dp, dns, fns in os.walk(root):
        level = dp.rstrip('/').count('/') - base
        print('  ' * level + os.path.basename(dp) + '/' + f'   [{len(dns)} dirs, {len(fns)} files]')
        for f in sorted(fns)[:per_dir]:
            print('  ' * (level + 1) + f, f'({os.path.getsize(os.path.join(dp, f)):,} B)')
        dns[:] = sorted(dns)[:per_dir] if level + 1 < depth else []
shallow(ROOT)
tables = [p for p in find('*.*', roots=(ROOT,), depth=3)
          if p.endswith(('.jsonl', '.json', '.csv', '.parquet')) and os.path.isfile(p)]
print('\nTABLE FILES:', tables[:30])
from graphpack.prepare_tasks import read_rows
for t in [t for t in tables if os.path.getsize(t) < 200_000_000][:4]:   # skip huge graph/embedding files
    try:
        rows = read_rows(t)
        if rows and isinstance(rows[0], dict):
            print(f'\n== {t}: {len(rows)} rows')
            for k, v in rows[0].items():
                print(f'   {k}: {str(v)[:200]!r}')
    except Exception as e:
        print('could not read', t, e)
for r in [p for p in find('*.md', roots=(ROOT,), depth=3) if 'readme' in p.lower()][:2]:
    print(f'\n===== {r} =====')
    print(open(r, encoding='utf-8', errors='replace').read()[:15000])

In [ ]:
# EDIT these three lines after looking at the inspect output
SRC = f'{COMP[0]}/dev/tasks.jsonl'                       # the task table
REPO_TEMPLATE = COMP[0] + '/dev/repos/{instance_id}'     # pre-patch snapshot per task
EXTRA = ''   # e.g. '--field-problem-statement issue --field-patch gold_patch'
!python -m graphpack.prepare_tasks --src {SRC} --repo-template "{REPO_TEMPLATE}" {EXTRA} --out tasks.jsonl
!wc -l tasks.jsonl

## 2. Retrieval-only baselines (CPU, minutes)

In [ ]:
for r in ['bm25', 'ppr', 'graphpack']:
    !python -m graphpack.run --tasks tasks.jsonl --out results/retrieval.jsonl --method retrieval --ranker {r}
!python -m graphpack.report results/*.jsonl --out-dir report

## 3. Serve Gemma 4 with vLLM
Point `MODEL_PATH` at the model you added as input (the folder with `config.json`).
Use the same checkpoint family the main competition allows (E4B, and E2B for the scale comparison).

In [ ]:
MODEL_PATH = [p for p in find('config.json', roots=('/kaggle/input',), depth=8) if 'gemma' in p.lower()
              and 'developer-agent' not in p]
MODEL_PATH_ALL = MODEL_PATH
print(MODEL_PATH)   # pick the E4B one below if several are listed
MODEL_PATH = os.path.dirname(next((p for p in MODEL_PATH if 'e4b' in p.lower()), MODEL_PATH[0]))   # EDIT if wrong
try:
    import vllm  # noqa
    print('vllm', vllm.__version__)
except ImportError:
    raise SystemExit('vLLM is not installed and L4 sessions are offline: attach the same vLLM input '
                     '(wheels/utility script) that the competition Getting Started notebook uses, then rerun')
def serve(model_path, name, port=8000, tp=1):
    # E2B/E4B fit on one 24 GB L4; tp=1 avoids head-count/tensor-parallel mismatches
    log = open(f'vllm_{name}.log', 'w')
    proc = subprocess.Popen(['vllm', 'serve', model_path, '--served-model-name', name, '--port', str(port),
                             '--max-model-len', '32768', '--tensor-parallel-size', str(tp),
                             '--gpu-memory-utilization', '0.90'], stdout=log, stderr=subprocess.STDOUT)
    import urllib.request
    for _ in range(180):
        try:
            urllib.request.urlopen(f'http://localhost:{port}/v1/models', timeout=2); break
        except Exception:
            time.sleep(5)
    else:
        raise RuntimeError(open(f'vllm_{name}.log').read()[-3000:])
    os.environ.update(LLM_BASE_URL=f'http://localhost:{port}/v1', LLM_API_KEY='EMPTY', LLM_MODEL=name)
    return proc
server = serve(MODEL_PATH, 'gemma4-e4b')

## 4. One-shot localization: budget × ranker × packing (main experiment)

In [ ]:
MODEL = 'gemma4-e4b'
GRID = [('bm25', 'bodies'), ('graphpack', 'bodies'), ('graphpack', 'skeleton'), ('graphpack', 'hybrid')]
for budget in [2000, 4000, 8000, 16000]:
    for ranker, mode in GRID:
        !python -m graphpack.run --tasks tasks.jsonl --out results/oneshot_{MODEL}.jsonl --method oneshot \
            --ranker {ranker} --pack {mode} --budget {budget} --model {MODEL}

## 5. Agents: grep tools vs graph tools vs graph tools + 2k starting pack

In [ ]:
for tools, sp in [('grep', 0), ('graph', 0), ('graph', 2000)]:
    !python -m graphpack.run --tasks tasks.jsonl --out results/agent_{MODEL}.jsonl --method agent \
        --tools {tools} --turns 15 --start-pack {sp} --context-limit 24000 --model {MODEL}

## 6. Scale: repeat the key configs with E2B
Stop the E4B server first, add the E2B model as input, then run.

In [ ]:
server.terminate(); server.wait()
E2B_PATH = [p for p in MODEL_PATH_ALL if 'e2b' in p.lower()]
print(E2B_PATH)
E2B_PATH = os.path.dirname(E2B_PATH[0])   # EDIT if the wrong one is picked
server = serve(E2B_PATH, 'gemma4-e2b')
MODEL = 'gemma4-e2b'
for budget in [2000, 4000, 8000, 16000]:
    for ranker, mode in [('bm25', 'bodies'), ('graphpack', 'hybrid')]:
        !python -m graphpack.run --tasks tasks.jsonl --out results/oneshot_{MODEL}.jsonl --method oneshot \
            --ranker {ranker} --pack {mode} --budget {budget} --model {MODEL}
for tools, sp in [('grep', 0), ('graph', 2000)]:
    !python -m graphpack.run --tasks tasks.jsonl --out results/agent_{MODEL}.jsonl --method agent \
        --tools {tools} --turns 15 --start-pack {sp} --model {MODEL}

## 7. Tables and figure for the paper
`report/results.md` has every table; `report/budget_curve.png` is Figure 1.
Config names do not contain the model, so report each model's files separately.

In [ ]:
for m in ['gemma4-e4b', 'gemma4-e2b']:
    files = ' '.join(glob.glob(f'results/*{m}.jsonl') + ['results/retrieval.jsonl'])
    !python -m graphpack.report {files} --out-dir report/{m}
!python -m graphpack.report results/retrieval.jsonl --out-dir report/retrieval
!cd /kaggle/working && zip -qr graphpack_results.zip results report && ls -la